<a href="https://colab.research.google.com/github/MengOonLee/LLM/blob/main/References/LangChain/LangChainAcademy/LangChain/Foundation/CreateAgent/ipynb/1.2_web_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%bash
apt install -y zstd pciutils lshw
curl -fsSL https://ollama.com/install.sh | sh
pip install --no-cache-dir -qU \
    langchain langgraph langchain-core \
    langchain-community langchain-ollama ollama \
    tavily-python

In [ ]:
!nohup ollama serve &

In [ ]:
!ollama pull gpt-oss:20b

## Without web search

In [2]:
import warnings
warnings.filterwarnings('ignore')
import dotenv

_ = dotenv.load_dotenv(dotenv_path=".env", override=True)

In [3]:
import langchain_ollama
from langchain import agents, messages
import time

model = langchain_ollama.ChatOllama(
    model="gpt-oss:20b",
    temperature=0.1,
    max_tokens=128
)

agent = agents.create_agent(
    model=model
)

question = messages.HumanMessage(content="""
    How up to date is your training knowledge?
""")

start_time = time.time()
response = agent.invoke(input={"messages": [question]})
end_time = time.time() - start_time
print(f"Time taken: %.2fs seconds"%(end_time))

for m in response["messages"]:
    m.pretty_print()

Time taken: 126.66s seconds
================================ Human Message =================================


    How up to date is your training knowledge?

================================== Ai Message ==================================

My training data goes up to **June 2024**. That means I can provide information, context, and insights that were publicly available up to that point. I don’t have access to events, developments, or new research that occurred after that date, and I can’t browse the web in real time. If you need the very latest updates, it’s a good idea to double‑check with current sources.


## Add web search tool

In [4]:
import tavily
import typing
from langchain import tools

tavily_client = tavily.TavilyClient()

@tools.tool
def web_search(query: str) -> typing.Dict[str, typing.Any]:
    """
    Search the web for information
    """

    return tavily_client.search(query=query)

web_search.invoke(input="""
    List of public holidays in Malaysia for 2026.
""")

{'query': 'List of public holidays in Malaysia for 2026.',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.eskimo.travel/en/blog/malaysia-public-holidays-2026?srsltid=AfmBOoo-7351gdxP5bKb8LCRfQHP9XtxvA1ES9wLl2luZlgrACov-Ucu',
   'title': 'Malaysia Public Holidays 2026: Calendar and Long Weekend Guide',
   'content': '# Malaysia Public Holidays 2026: Complete Calendar and Festival Traditions. The **2026 public holiday Malaysia** calendar features a diverse mix of cultural, religious and national celebrations observed across the country. Malaysia’s rich multicultural identity means the year includes major festivals like **Chinese New Year (CNY 2026)**, **Hari Raya Aidilfitri**, **Hari Raya Haji**, and Deepavali, alongside national events such as **Merdeka Day** and Labour Day. The **public holiday Malaysia** schedule in 2026 offers several natural long weekends ideal for domestic or regional travel. As the **2026 public holiday Malaysia** c

In [6]:
import tavily
import typing
import langchain_ollama
from langchain import tools, agents, messages
import time

tavily_client = tavily.TavilyClient()

@tools.tool
def web_search(query: str) -> typing.Dict[str, typing.Any]:
    """
    Search the web for information
    """

    return tavily_client.search(query=query)

model = langchain_ollama.ChatOllama(
    model="gpt-oss:20b",
    temperature=0.1,
    max_tokens=128
)

system_prompt = """
    You are a helpful assistant.
    Please keep the below structure.

    Country: The country of the holiday.
    Date: The date of the holiday.
    Name: The name of the holiday.
"""

agent = agents.create_agent(
    model=model,
    tools=[web_search],
    system_prompt=system_prompt
)

question = messages.HumanMessage(content="""
    List of public holidays in Malaysia for 2026.
""")

start_time = time.time()
response = agent.invoke(input={"messages": [question]})
end_time = time.time() - start_time
print(f"Time taken: %.2fs seconds"%(end_time))

for m in response["messages"]:
    m.pretty_print()

Time taken: 97.99s seconds
================================ Human Message =================================


    List of public holidays in Malaysia for 2026.

================================== Ai Message ==================================
Tool Calls:
  web_search (11fd93f4-0ddc-4de6-a891-dc575eccca4f)
 Call ID: 11fd93f4-0ddc-4de6-a891-dc575eccca4f
  Args:
    query: Malaysia public holidays 2026 list
================================= Tool Message =================================
Name: web_search

{"query": "Malaysia public holidays 2026 list", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.timeanddate.com/holidays/malaysia/2026", "title": "Holidays and Observances in Malaysia in 2026", "content": "| Feb 2 | Monday | Day off for Thaipusam | State Holiday | JHR, KDH, KUL, NSN, PJY, PNG, PRK, SGR |. | Feb 19 | Thursday | First Day of Ramadan (Tentative Date) | State Holiday | JHR, KDH, MLK |. | Mar 21 | Saturday | Hari Raya Puasa Day 2 (Ten